# NullVector Progress Notebook

Phase 04 / Spec 02: Metadata-Driven Document Selection Before Retrieval

Prerequisites:
- Execute from the repository root with project dependencies available (for example via `uv run jupyter lab`).
- This notebook uses the filesystem backend and a noop gateway, so it does not require PostgreSQL or live provider credentials.

In [ ]:
# environment setup
from pathlib import Path
import json
import shutil

PROJECT_ROOT = Path.cwd()
ARTIFACT_ROOT = PROJECT_ROOT / 'tmp' / 'progress-spec02'
if ARTIFACT_ROOT.exists():
    shutil.rmtree(ARTIFACT_ROOT)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'artifact_root={ARTIFACT_ROOT}')

In [ ]:
# imports
from nullvector.domain.document_selection import (
    DocumentFilterClause,
    DocumentFilterOperator,
    DocumentMetadataRecord,
    MetadataSelectionPlan,
    MetadataSelectionPlannerRequest,
    MetadataSelectionRequest,
)
from nullvector.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayService,
    NoopProviderAdapter,
    NoopScriptedResponse,
)
from nullvector.retrieval import MetadataSelectionPlanner, MetadataSelectionService
from nullvector.storage import FilesystemStorageConfig

In [ ]:
# configuration
COLLECTION_ID = 'collection-progress'
FIELDS = ('company', 'year', 'case_type')
METADATA_RECORDS = (
    DocumentMetadataRecord(
        document_id='doc-001',
        display_name='Acme Annual Report 2024',
        attributes={'company': 'Acme', 'year': 2024, 'case_type': 'report'},
    ),
    DocumentMetadataRecord(
        document_id='doc-002',
        display_name='Acme Board Memo 2023',
        attributes={'company': 'Acme', 'year': 2023, 'case_type': 'memo'},
    ),
    DocumentMetadataRecord(
        document_id='doc-003',
        display_name='Beta Litigation 2024',
        attributes={'company': 'Beta', 'year': 2024, 'case_type': 'litigation'},
    ),
)

storage = FilesystemStorageConfig(root=str(ARTIFACT_ROOT))
service = MetadataSelectionService(storage=storage)
planner = MetadataSelectionPlanner()
planner_gateway = GatewayService(
    GatewayConfig(
        default_model='test-model',
        audit=GatewayAuditConfig(persist_root=str(ARTIFACT_ROOT / 'audit')),
    ),
    provider_adapter=NoopProviderAdapter(
        {
            'metadata_selection_plan': NoopScriptedResponse(
                output_json={
                    'normalized_query': 'acme 2024 report',
                    'clauses': [
                        {'field': 'company', 'operator': 'eq', 'value': 'Acme'},
                        {'field': 'year', 'operator': 'gte', 'value': 2024},
                    ],
                    'reasoning_summary': 'Use company and publication year to narrow the collection.',
                }
            )
        }
    ),
)

In [ ]:
# execution
explicit_plan = MetadataSelectionPlan(
    raw_query='Acme 2024 report',
    normalized_query='acme 2024 report',
    clauses=(
        DocumentFilterClause(
            field='company',
            operator=DocumentFilterOperator.EQ,
            value='Acme',
        ),
        DocumentFilterClause(
            field='year',
            operator=DocumentFilterOperator.GTE,
            value=2024,
        ),
    ),
    reasoning_summary='Filter to Acme documents from 2024 onward.',
)

explicit_response = service.select(
    MetadataSelectionRequest(
        collection_id=COLLECTION_ID,
        selection_run_id='metadata-selection-explicit',
        plan=explicit_plan,
        allowed_fields=FIELDS,
        metadata_records=METADATA_RECORDS,
        limit=5,
    )
)
tuple(candidate.document_id for candidate in explicit_response.candidates)

In [ ]:
# execution
planned_plan = planner.plan(
    MetadataSelectionPlannerRequest(
        query='Acme 2024 report',
        allowed_fields=FIELDS,
        field_descriptions={
            'company': 'Issuer or company name',
            'year': 'Publication year',
            'case_type': 'Document category',
        },
    ),
    gateway=planner_gateway,
)

planned_response = service.select(
    MetadataSelectionRequest(
        collection_id=COLLECTION_ID,
        selection_run_id='metadata-selection-planned',
        plan=planned_plan,
        allowed_fields=FIELDS,
        metadata_records=METADATA_RECORDS,
        limit=5,
    )
)
tuple(candidate.document_id for candidate in planned_response.candidates)

In [ ]:
# inspect results
summary = {
    'explicit_candidate_ids': [candidate.document_id for candidate in explicit_response.candidates],
    'planned_candidate_ids': [candidate.document_id for candidate in planned_response.candidates],
    'explicit_artifacts': {
        'metadata_index_path': explicit_response.metadata_index_path,
        'selection_plan_path': explicit_response.selection_plan_path,
        'selection_results_path': explicit_response.selection_results_path,
    },
    'planned_plan': planned_plan.model_dump(mode='json'),
}
print(json.dumps(summary, indent=2, sort_keys=True))

### Known Limitations

- This notebook exercises the filesystem backend only; PostgreSQL-backed metadata indexing is implemented but not demonstrated here.
- The natural-language planner uses a noop gateway script for deterministic execution.
- Metadata selection narrows candidate documents only; it does not run retrieval or QA.